In [1]:
from pathlib import Path
import json
import random
import time
import gc

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

from peft import PeftModel

[HAMI-core Msg(590:139995016449344:libvgpu.c:839)]: Initializing.....


In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA A100 80GB PCIe


[HAMI-core Msg(590:139995016449344:libvgpu.c:855)]: Initialized


In [3]:
PROJECT_DIR = Path(
    "/home/jovyan/project work/data_analyssis/fine tuning"
)

BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

ADAPTER_DIR = (
    PROJECT_DIR
    / "outputs"
    / "mistral_qlora"
    / "final_adapter"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "outputs"
    / "results"
    / "generator_evaluation"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Adapter directory:", ADAPTER_DIR)
print("Results directory:", RESULTS_DIR)

Project directory: /home/jovyan/project work/data_analyssis/fine tuning
Adapter directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/mistral_qlora/final_adapter
Results directory: /home/jovyan/project work/data_analyssis/fine tuning/outputs/results/generator_evaluation


In [4]:
print("Adapter exists:", ADAPTER_DIR.exists())

if ADAPTER_DIR.exists():
    print("Adapter files:")
    for file_path in sorted(ADAPTER_DIR.iterdir()):
        print(" -", file_path.name)

Adapter exists: True
Adapter files:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


In [5]:
SEED = 42

random.seed(SEED)
set_seed(SEED)

print("Seed:", SEED)

Seed: 42


In [6]:
CLASS_DESCRIPTIONS = {
    2: {
        "label": "Religious hate speech",
        "definition": (
            "A post that insults, humiliates, stereotypes, or expresses "
            "hostility towards a person or group because of religion, "
            "sect, or religious identity."
        ),
        "boundary": (
            "The hostility must be based specifically on religion, sect, "
            "or religious identity. Do not generate general profanity or "
            "gender-based abuse as the main content."
        ),
    },

    3: {
        "label": "Sexist abusive language",
        "definition": (
            "A post that insults, humiliates, stereotypes, or degrades "
            "a person because of gender."
        ),
        "boundary": (
            "The abuse must be connected specifically to gender. Do not "
            "generate general profanity or religious hostility as the "
            "main content."
        ),
    },

    4: {
        "label": "Profane language",
        "definition": (
            "A post containing vulgar, obscene, or strongly offensive "
            "language, without necessarily targeting religion or gender."
        ),
        "boundary": (
            "The post may contain vulgar or obscene language, but religion "
            "or gender must not be the main reason for the abuse."
        ),
    },
}

In [7]:
def build_training_style_prompt(class_id):
    """
    Build a zero-shot prompt similar to the instruction used during
    supervised fine-tuning.
    """

    if class_id not in CLASS_DESCRIPTIONS:
        raise ValueError(
            f"Unsupported class_id: {class_id}. "
            f"Expected one of {list(CLASS_DESCRIPTIONS.keys())}."
        )

    class_label = CLASS_DESCRIPTIONS[class_id]["label"]
    class_definition = CLASS_DESCRIPTIONS[class_id]["definition"]

    user_prompt = f"""
You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class:
{class_label}

Class definition:
{class_definition}

Generate one natural, informal Roman Urdu social-media post that belongs to the target class.

Write primarily in Roman Urdu using the English Latin alphabet. Natural English code-mixing is acceptable.

Return only the generated post.
""".strip()

    return [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

In [8]:
def build_strong_zero_shot_prompt(class_id):
    """
    Build a stricter zero-shot prompt with explicit class boundaries.
    """

    if class_id not in CLASS_DESCRIPTIONS:
        raise ValueError(
            f"Unsupported class_id: {class_id}. "
            f"Expected one of {list(CLASS_DESCRIPTIONS.keys())}."
        )

    class_info = CLASS_DESCRIPTIONS[class_id]

    user_prompt = f"""
You are an expert data synthesis assistant for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class:
{class_info["label"]}

Class definition:
{class_info["definition"]}

Class boundary:
{class_info["boundary"]}

Generate exactly ONE new social-media post that clearly belongs to the target class.

Requirements:
- Write primarily in Roman Urdu using the English Latin alphabet.
- Do not use Urdu, Arabic, or Devanagari script.
- Natural English code-mixing is allowed.
- Use an informal social-media style.
- Express one coherent idea.
- Do not explain the label or mention the dataset.
- Do not include headings, notes, quotation marks, or multiple alternatives.
- Do not copy or closely paraphrase a dataset example.

Return only the generated post.
""".strip()

    return [
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

In [9]:
for class_id in CLASS_DESCRIPTIONS:
    print("=" * 80)
    print("CLASS:", CLASS_DESCRIPTIONS[class_id]["label"])
    print("=" * 80)

    prompt = build_training_style_prompt(class_id)
    print(prompt[0]["content"])
    print()

CLASS: Religious hate speech
You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class:
Religious hate speech

Class definition:
A post that insults, humiliates, stereotypes, or expresses hostility towards a person or group because of religion, sect, or religious identity.

Generate one natural, informal Roman Urdu social-media post that belongs to the target class.

Write primarily in Roman Urdu using the English Latin alphabet. Natural English code-mixing is acceptable.

Return only the generated post.

CLASS: Sexist abusive language
You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class:
Sexist abusive language

Class definition:
A post that insults, humiliates, stereotypes, or degrades a person because of gender.

Generate one natural, informal Roman Urdu social-media post that belongs to the target class.

Write primarily in

In [10]:
for class_id in CLASS_DESCRIPTIONS:
    print("=" * 80)
    print("CLASS:", CLASS_DESCRIPTIONS[class_id]["label"])
    print("=" * 80)

    prompt = build_strong_zero_shot_prompt(class_id)
    print(prompt[0]["content"])
    print()

CLASS: Religious hate speech
You are an expert data synthesis assistant for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class:
Religious hate speech

Class definition:
A post that insults, humiliates, stereotypes, or expresses hostility towards a person or group because of religion, sect, or religious identity.

Class boundary:
The hostility must be based specifically on religion, sect, or religious identity. Do not generate general profanity or gender-based abuse as the main content.

Generate exactly ONE new social-media post that clearly belongs to the target class.

Requirements:
- Write primarily in Roman Urdu using the English Latin alphabet.
- Do not use Urdu, Arabic, or Devanagari script.
- Natural English code-mixing is allowed.
- Use an informal social-media style.
- Express one coherent idea.
- Do not explain the label or mention the dataset.
- Do not include headings, notes, quotation marks, or multiple alternatives.
- Do not copy

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Padding side:", tokenizer.padding_side)

Tokenizer loaded.
Pad token: </s>
EOS token: </s>
Padding side: left


In [12]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured.")

4-bit quantization configured.


In [13]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)

base_model.eval()

print("Base Mistral loaded.")
print("Device:", next(base_model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base Mistral loaded.
Device: cuda:0


In [14]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

finetuned_model.eval()

print("Fine-tuned LoRA adapter loaded.")
print("Model type:", type(finetuned_model).__name__)

Fine-tuned LoRA adapter loaded.
Model type: PeftModelForCausalLM


In [15]:
print(finetuned_model)

if hasattr(finetuned_model, "active_adapters"):
    print("Active adapter:", finetuned_model.active_adapters)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [18]:
@torch.inference_mode()
def generate_one(
    model,
    messages,
    seed=42,
    max_new_tokens=80,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15
):
    set_seed(seed)

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    model_inputs = {
        key: value.to(model.device)
        for key, value in model_inputs.items()
    }

    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    prompt_length = model_inputs["input_ids"].shape[-1]

    generated_ids = output_ids[0, prompt_length:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return generated_text

In [19]:
class_id = 3

messages = build_training_style_prompt(class_id)

output = generate_one(
    model=finetuned_model,
    messages=messages,
    seed=42
)

print("Target class:", CLASS_DESCRIPTIONS[class_id]["label"])
print("Generated output:")
print(output)

Target class: Sexist abusive language
Generated output:
teri maa ki choot marunga koi baat nai randi kutta


In [20]:
for seed in [42, 43, 44, 45, 46]:
    output = generate_one(
        model=finetuned_model,
        messages=build_training_style_prompt(3),
        seed=seed
    )

    print(f"\nSeed {seed}")
    print(output)


Seed 42
teri maa ki choot marunga koi baat nai randi kutta

Seed 43
bhaunk bharwe 😂

Seed 44
hijra ki kya baat hai?

Seed 45
randi k bacha ye 🐕

Seed 46
tere maa ka yaar chod k bhaunk raha hai na hijra


In [21]:
print(hasattr(finetuned_model, "disable_adapter"))

True


In [22]:
@torch.inference_mode()
def compare_base_and_finetuned(
    class_id,
    seeds=(42, 43, 44, 45, 46),
    prompt_builder=build_training_style_prompt,
):
    messages = prompt_builder(class_id)

    rows = []

    for seed in seeds:
        # Base Mistral: temporarily disable LoRA
        with finetuned_model.disable_adapter():
            base_output = generate_one(
                model=finetuned_model,
                messages=messages,
                seed=seed,
            )

        # Fine-tuned Mistral: LoRA active
        finetuned_output = generate_one(
            model=finetuned_model,
            messages=messages,
            seed=seed,
        )

        rows.append(
            {
                "class_id": class_id,
                "class_label": CLASS_DESCRIPTIONS[class_id]["label"],
                "seed": seed,
                "base_output": base_output,
                "finetuned_output": finetuned_output,
            }
        )

    return pd.DataFrame(rows)

In [23]:
sexism_comparison = compare_base_and_finetuned(
    class_id=3,
    seeds=[42, 43, 44, 45, 46],
)

pd.set_option("display.max_colwidth", None)
display(sexism_comparison)

,class_id,class_label,seed,base_output,finetuned_output
0,3,Sexist abusive language,42,اِس لدیون کے بھائی خود کو تو مہربانہ بناتे ہوئیں، لوگوں کی وجہ سے تمہین کا چارکہ,teri maa ki choot marunga koi baat nai randi kutta
1,3,Sexist abusive language,43,تو بہترین لدھریاں ہیں، جسکے طور پر خود بنائی ہوئی گلدیشوں کی بیٹیوں سے مزید عظمت,bhaunk bharwe 😂
2,3,Sexist abusive language,44,اِس مेری بھیجوں کے باغ پر چلتی ہوئی دونوں خاندان کی اُمرانات کا ساتھ بیٹوں سے حد,hijra ki kya baat hai?
3,3,Sexist abusive language,45,تو ملکوں سائین ہوتے ہوئے جسم کا شبوحات چھوتا رہے، تو مężczyźni ہوتے ہوئے تک کہ آپ خدا,randi k bacha ye 🐕
4,3,Sexist abusive language,46,اِس لَدْدے\_بھیجوں تُو مmeroون کہ تم خود بناؤ، لڑکاوں نے آپ کا شرف چھوड़ا ہے ـ‑‑ـ „is,tere maa ka yaar chod k bhaunk raha hai na hijra


In [25]:
comparison_path = RESULTS_DIR / "sexism_base_vs_finetuned_zero_shot.csv"

sexism_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", comparison_path)

Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/results/generator_evaluation/sexism_base_vs_finetuned_zero_shot.csv


In [29]:
CLEANED_TRAIN_PATH = (
    PROJECT_DIR
    / "outputs"
    / "cleaned_ruhsold_train.csv"
)

train_df = pd.read_csv(CLEANED_TRAIN_PATH)

print("Dataset shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

display(train_df.head())

Dataset shape: (6401, 2)
Columns: ['text', 'label']


,text,label
0,kia howa hai aap ko allah bless and protect you aameen,1
1,randdi hai,3
2,smjh to agai thi mjhy,1
3,haan yrr tuny sahi thukayi ki abhi tak lund dard kr raha hai sahi randi hai...wesy ghalti ki tjhy lund dy k abb pta nhi lund ko koi bemari he naw llag jaye gandi phudi k moun jo lag gayi hai😈😈😈👙🤣🤣,3
4,rundi ka bacha bharwaaa ptm ka kuttaa,0


In [30]:
print(train_df["label"].value_counts().sort_index())

label
0    1534
1    3422
2     500
3     535
4     410
Name: count, dtype: int64


In [31]:
sexism_examples = (
    train_df.loc[train_df["label"] == 3, "text"]
    .sample(n=30, random_state=42)
)

for index, text in enumerate(sexism_examples, start=1):
    print(f"{index}. {text}")

1. gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,
2. yar is pagal ko dafa karo islam ko badnam kr rahi ha randi kahin ki.ak video men nangi ho jati or agli video men hajan ban jati ha
3. sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho
4. bc nasal ki randi
5. to be na randi hy na asle....shame on u
6. abu se bhai jaan se chacha jaan sabhi se to halala krwa leti hai begum  or ye tera jeeja ravish khan usse bhi  salon humse akele he kar denge
7. #rashamidesai tu randi moti apni maa ko bech chuki hai ...kisi aur ke hatho..to apni awkat ne rah kar game plan kar ...arhan to vaduaa hai...ram mandir ban rha hai na..but tu chud rahi hai kisi aur ka bister garam kar rhi hai
8. randi tere khene se koch nahi hoga kanjar nasal ki payedawar
9. yeh dekh chakki union ki randi
10. zartaj gul tu mera lora chup ghastiyan lully khanay wali ghashti imran niazi ka lun muh mein leti hai phudi on ki teri maa ko lun maru ghashti .....teri ph

In [32]:
def sample_demonstrations(
    class_id,
    n_examples=5,
    random_state=42,
):
    class_examples = train_df.loc[
        train_df["label"] == class_id,
        "text"
    ].dropna().astype(str)

    if len(class_examples) < n_examples:
        raise ValueError(
            f"Class {class_id} has only {len(class_examples)} examples."
        )

    return class_examples.sample(
        n=n_examples,
        random_state=random_state,
    ).tolist()

In [33]:
sexism_demos = sample_demonstrations(
    class_id=3,
    n_examples=5,
    random_state=42,
)

for i, example in enumerate(sexism_demos, start=1):
    print(f"{i}. {example}")

1. gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,
2. yar is pagal ko dafa karo islam ko badnam kr rahi ha randi kahin ki.ak video men nangi ho jati or agli video men hajan ban jati ha
3. sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho
4. bc nasal ki randi
5. to be na randi hy na asle....shame on u


In [34]:
def build_few_shot_prompt(
    class_id,
    demonstrations,
):
    class_info = CLASS_DESCRIPTIONS[class_id]

    formatted_examples = "\n".join(
        f"Example {i}: {example}"
        for i, example in enumerate(demonstrations, start=1)
    )

    instruction = f"""You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class: {class_info["label"]}
Class definition: {class_info["definition"]}
Class boundary: {class_info["boundary"]}

Here are real examples from the target class:

{formatted_examples}

Generate one new natural, informal Roman Urdu social-media post that clearly belongs to the target class.

Requirements:
- Write in Roman Urdu using the Latin alphabet.
- English code-mixing is allowed when natural.
- Generate only one post.
- Do not copy or closely paraphrase any example.
- The post itself must express the target category rather than discuss or condemn it.
- Do not produce Urdu or Arabic script.
- Return only the generated post without a label, explanation, quotation marks, or additional text."""

    return [
        {
            "role": "user",
            "content": instruction,
        }
    ]

In [35]:
sexism_few_shot_messages = build_few_shot_prompt(
    class_id=3,
    demonstrations=sexism_demos,
)

print(sexism_few_shot_messages[0]["content"])

You are generating synthetic examples for an academic Roman Urdu hate-speech and abusive-language classification dataset.

Target class: Sexist abusive language
Class definition: A post that insults, humiliates, stereotypes, or degrades a person because of gender.
Class boundary: The abuse must be connected specifically to gender. Do not generate general profanity or religious hostility as the main content.

Here are real examples from the target class:

Example 1: gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,
Example 2: yar is pagal ko dafa karo islam ko badnam kr rahi ha randi kahin ki.ak video men nangi ho jati or agli video men hajan ban jati ha
Example 3: sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho
Example 4: bc nasal ki randi
Example 5: to be na randi hy na asle....shame on u

Generate one new natural, informal Roman Urdu social-media post that clearly belongs to the target class.

Requirements:
- Write in

In [36]:
@torch.inference_mode()
def compare_few_shot_models(
    class_id,
    seeds=(42, 43, 44, 45, 46),
    n_demonstrations=5,
    demonstration_seed=42,
):
    demonstrations = sample_demonstrations(
        class_id=class_id,
        n_examples=n_demonstrations,
        random_state=demonstration_seed,
    )

    messages = build_few_shot_prompt(
        class_id=class_id,
        demonstrations=demonstrations,
    )

    rows = []

    for seed in seeds:
        # Base Mistral: LoRA adapter disabled
        with finetuned_model.disable_adapter():
            base_output = generate_one(
                model=finetuned_model,
                messages=messages,
                seed=seed,
            )

        # Fine-tuned Mistral: LoRA adapter enabled
        finetuned_output = generate_one(
            model=finetuned_model,
            messages=messages,
            seed=seed,
        )

        rows.append(
            {
                "class_id": class_id,
                "class_label": CLASS_DESCRIPTIONS[class_id]["label"],
                "prompt_type": "few_shot",
                "n_demonstrations": n_demonstrations,
                "demonstration_seed": demonstration_seed,
                "generation_seed": seed,
                "demonstrations": " ||| ".join(demonstrations),
                "base_output": base_output,
                "finetuned_output": finetuned_output,
            }
        )

    return pd.DataFrame(rows)

In [37]:
sexism_few_shot_comparison = compare_few_shot_models(
    class_id=3,
    seeds=[42, 43, 44, 45, 46],
    n_demonstrations=5,
    demonstration_seed=42,
)

display(
    sexism_few_shot_comparison[
        [
            "generation_seed",
            "base_output",
            "finetuned_output",
        ]
    ]
)

,generation_seed,base_output,finetuned_output
0,42,"tere mamu ne beti ko sharmiya dikhai di, uski wajah se tumhara aadmi zyada accha hai?\n(Your uncle showed off his daughter, so your man is better than hers?)",teri behno ki kutti
1,43,"tumhari maa ne apne betey ko ladki banay thi, par wo apni taraf se woh bani thi bechara, ab woh is liyec bina hai ghar se rah gayi, baki logon ke saath hoon main, meri dostoun ke saath, uska gulam nah",shukla bharwe akhrot ki gashti orat
2,44,"tere dam dam liyain bhai logon ne, tumhen gandum ke saath woh baithne dene waale hain, kyun ki tum hen khwahisheen-dil hain, meray pas banane wali randian hain.",bc hmara mard hai kya? apni behno ko ghairat se rakh lo.
3,45,"Here's a synthetic Roman Urdu social media post that fits the target class:\n\ntumhari maa ne meri beti ke saath ghar pe rakhne se pehle aayi thi, magar usse aage tumhein dikhane wali bint-e-kauar hain!\n\n(Your mother used",ye bhenchod ajeeb saali
4,46,"tere mamme ke saath rehne wale logon se tumhari aurat bano, kyunki tum apni aukad bhar sakti hai, lekin wo logon ne apni aukad bharane mein aapko zulm karte hain, tumhara jaanab farq nahi pa",bhai tu toh saali hy kyuki tu kisi aur sy pyar kar rha ha


In [39]:
few_shot_path = (
    RESULTS_DIR
    / "sexism_base_vs_finetuned_few_shot.csv"
)

sexism_few_shot_comparison.to_csv(
    few_shot_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", few_shot_path)

Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/results/generator_evaluation/sexism_base_vs_finetuned_few_shot.csv


In [42]:
from difflib import SequenceMatcher

def similarity_ratio(text_a, text_b):
    return SequenceMatcher(
        None,
        str(text_a).lower(),
        str(text_b).lower(),
    ).ratio()


for _, row in sexism_few_shot_comparison.iterrows():
    output = row["finetuned_output"]

    scores = [
        similarity_ratio(output, demo)
        for demo in sexism_demos
    ]

    best_index = max(
        range(len(scores)),
        key=scores.__getitem__,
    )

    best_score = scores[best_index]

    print("=" * 90)
    print("Output:", output)
    print("Closest demonstration:", sexism_demos[best_index])
    print("Similarity:", round(best_score, 3))

Output: teri behno ki kutti
Closest demonstration: bc nasal ki randi
Similarity: 0.389
Output: shukla bharwe akhrot ki gashti orat
Closest demonstration: sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap  kya cheez ho
Similarity: 0.314
Output: bc hmara mard hai kya? apni behno ko ghairat se rakh lo.
Closest demonstration: gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,
Similarity: 0.368
Output: ye bhenchod ajeeb saali
Closest demonstration: bc nasal ki randi
Similarity: 0.35
Output: bhai tu toh saali hy kyuki tu kisi aur sy pyar kar rha ha
Closest demonstration: to be na randi hy na asle....shame on u
Similarity: 0.333


In [43]:
zero_shot_finetuned = sexism_comparison[
    ["seed", "finetuned_output"]
].copy()

zero_shot_finetuned = zero_shot_finetuned.rename(
    columns={
        "seed": "generation_seed",
        "finetuned_output": "zero_shot_output",
    }
)


few_shot_finetuned = sexism_few_shot_comparison[
    ["generation_seed", "finetuned_output"]
].copy()

few_shot_finetuned = few_shot_finetuned.rename(
    columns={
        "finetuned_output": "few_shot_output",
    }
)


finetuned_prompt_comparison = zero_shot_finetuned.merge(
    few_shot_finetuned,
    on="generation_seed",
    how="inner",
)

display(finetuned_prompt_comparison)

,generation_seed,zero_shot_output,few_shot_output
0,42,teri maa ki choot marunga koi baat nai randi kutta,teri behno ki kutti
1,43,bhaunk bharwe 😂,shukla bharwe akhrot ki gashti orat
2,44,hijra ki kya baat hai?,bc hmara mard hai kya? apni behno ko ghairat se rakh lo.
3,45,randi k bacha ye 🐕,ye bhenchod ajeeb saali
4,46,tere maa ka yaar chod k bhaunk raha hai na hijra,bhai tu toh saali hy kyuki tu kisi aur sy pyar kar rha ha


In [44]:
from difflib import SequenceMatcher

def similarity_ratio(text_a, text_b):
    return SequenceMatcher(
        None,
        str(text_a).lower(),
        str(text_b).lower(),
    ).ratio()


def get_best_demo_match(output, demonstrations):
    scores = [
        similarity_ratio(output, demo)
        for demo in demonstrations
    ]

    best_index = max(
        range(len(scores)),
        key=scores.__getitem__,
    )

    return {
        "closest_demo": demonstrations[best_index],
        "character_similarity": scores[best_index],
    }


match_results = sexism_few_shot_comparison[
    "finetuned_output"
].apply(
    lambda output: get_best_demo_match(
        output,
        sexism_demos,
    )
)

sexism_few_shot_comparison["closest_demo"] = match_results.apply(
    lambda result: result["closest_demo"]
)

sexism_few_shot_comparison["character_similarity"] = match_results.apply(
    lambda result: result["character_similarity"]
)

display(
    sexism_few_shot_comparison[
        [
            "generation_seed",
            "finetuned_output",
            "closest_demo",
            "character_similarity",
        ]
    ]
)

,generation_seed,finetuned_output,closest_demo,character_similarity
0,42,teri behno ki kutti,bc nasal ki randi,0.388889
1,43,shukla bharwe akhrot ki gashti orat,sir gazhi hijra hy ap bhe dp laga lein pta tu chly ap kya cheez ho,0.313725
2,44,bc hmara mard hai kya? apni behno ko ghairat se rakh lo.,"gashti orat tujhy koi amreci ni mila abi tak jo teri garmi ko thik kry kise gashti maa ki baiti,",0.368421
3,45,ye bhenchod ajeeb saali,bc nasal ki randi,0.350000
4,46,bhai tu toh saali hy kyuki tu kisi aur sy pyar kar rha ha,to be na randi hy na asle....shame on u,0.333333


In [45]:
sexism_few_shot_comparison["character_similarity"] = (
    sexism_few_shot_comparison["character_similarity"]
    .round(3)
)

In [46]:
few_shot_path = (
    RESULTS_DIR
    / "sexism_base_vs_finetuned_few_shot.csv"
)

sexism_few_shot_comparison.to_csv(
    few_shot_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", few_shot_path)

Saved to: /home/jovyan/project work/data_analyssis/fine tuning/outputs/results/generator_evaluation/sexism_base_vs_finetuned_few_shot.csv
